## **FindMyText**: Step-by-step example

**FindMyText** is a Python package which can be used to find out whether a given text appears, in part or in full, within a text corpus. In other words, the purpose of **FindMyText** is to efficiently detect _text containment_, and is specifically designed to satisfy the three following criteria:
 
1. The method should be *robust* to small discrepancies between the texts (due to e.g. text normalisation, formatting variants, etc.)
2. The method should *scale* to large, web-crawled corpora 
3. And it should focus on detecting occurrences of _shared text segments_, rather than mere similarity

A central use case for **FindMyText** is to determine whether a large corpus contains texts that may be copyrighted or breach licensing conditions. The tool builds upon existing techniques for document fingerprinting, but with a new approach to precisely detect the presence of fingerprint chains. 

### Getting the data

In this short example, we will use a collection of 50 Italian-language books from the Gutenberg Project. They can be directly loaded from HuggingFace:

In [37]:
from datasets import load_dataset
corpus = load_dataset("efederici/gutenberg-it")
corpus = corpus["train"].select(range(50))  # Limit to first 50 books for demo purposes

Those books are about 300-400 pages in length on average:

In [38]:
avg_chars = sum(len(text["text"]) for text in corpus) // len(corpus) # type:ignore
print(f"Average book length: {avg_chars:,} characters")

Average book length: 469,945 characters



### Building the index

The first step of **FindMyText** is to extract the set of _fingerprints_ for each text of the corpus. Document fingerprints are essentially $k$-grams extracted from the text and then converted to hashes. The fingerprints are then stored in an inverted index, which maps every fingerprints to a list of (`doc_id`, `position`) pairs indicating where the fingerprint has been observed in the corpus. To avoid storing the entire set of fingerprints, **FindMyText** relies on a technique called *winnowing*.

The extraction of fingerprints can be parallelised, like this: 

In [39]:
from findmytext import index_builder
files = index_builder.index_data_parallel(corpus, "gutenberg_temp", n_workers=4)


And saving index files to gutenberg_temp
Indexing 50 documents across 4 workers...
  indexed -> gutenberg_temp/final-worker01.jsonl.gz
  indexed -> gutenberg_temp/final-worker03.jsonl.gz
  indexed -> gutenberg_temp/final-worker04.jsonl.gz
  indexed -> gutenberg_temp/final-worker02.jsonl.gz


The output of this call is a set of 4 files containing the fingerprints and their exact locations in the corpus. 

The only thing that remains is to merge those temporary files into a single index. We will store this final index in the directory `gutenberg_index`: 

In [40]:
index_builder.merge_indexes_from_dir("gutenberg_temp", "gutenberg_index")

# Once the index is built, you can remove the directory with the temporary index files
!rm -rf gutenberg_temp

Merging the following index files: ['gutenberg_temp/final-worker03.jsonl.gz', 'gutenberg_temp/final-worker02.jsonl.gz', 'gutenberg_temp/final-worker04.jsonl.gz', 'gutenberg_temp/final-worker01.jsonl.gz']
Saving merged index to gutenberg_index
Meta parameters: {'length': 4, 'window_size': 6, 'base': 256, 'punctuation': False}


  Found 50 unique documents.
Saved document ID mapping for 50 documents.
Processed 1,000,000 fingerprints...
Saving final index (1,030,202 fingerprints)...Done


The constructed index is designed for scalability and efficiency. In particular, it is disk-based (through the use of memory-mapped arrays), making it possible to work with large corpora for which the index of fingerprints cannot be stored in memory. 

### Detecting text containment

Let's now use our index to detect which texts are or are not included in the corpus. We'll start by taking a small excerpt of our corpus, here a few lines of Italian poetry:

In [41]:
excerpt = str(corpus[5]["text"][5078:6287])
print(excerpt)

Già di cavalli e più di cavalieri
Tagliati e morti v'è copia sì grande,
Che alzar se ne potrìano i monti intieri;
Onde convien che il resto si disbande,
Ed alla fuga dassi volentieri.
Ricciardo di piacer lagrime spande,
E seco gli altri due fanno lo stesso,
E van correndo alle lor dame appresso

14

Ma non sì tosto giunsero là dove
Il Cavalier del Pianto egro giacea,
Che seppero l'acerbe e triste nuove,
E chiamaron Fortuna iniqua e rea,
Tiranno il Fato, e dispietato Giove.
Prese Ricciardo, conforme potea,
Il cavalier ferito e mezzo morto
In su le spalle, e lo condusse al porto;

[13]

15

E mentre un buon cerusico lo cura,
Domanda all'oste il mesto Ricciardetto,
Qual sia del vecchio rege la natura,
Per sapere qual possa avere effetto
Delle tre donne l'acerba cattura.
Rispose l'oste: Egli è un uom maladetto
Che sta con gli demonj e gli aversieri
Tutte le notti e tutti i giorni intieri:

16

Ed ora li fa fare il muratore,
Ed ora il fabbro, ed ora il legnajuolo;
Chè fabbricar gli ho visto

The most straightforward way to detect text containment is by counting the number of shared fingerprints between the query and the indexed documents. The more fingerprints they share, the more likely it is that the query text is contained within the document.

To do this detection, we first load a `NbSharedFingerprintsDetector`, which takes as input the directory containing the index: 

In [42]:
from findmytext import detectors
baseline_detector = detectors.NbSharedFingerprintsDetector("gutenberg_index")


The detection can then be performed by calling `get_containment_scores`, which takes a text as input and returns a dictionary mapping document IDs to text containment scores:

In [43]:
print("Detecting verbatim excerpt in corpus based on number of shared fingerprints")
scores = baseline_detector.get_containment_scores(excerpt)
print("Scores:", scores)

correct_doc_id = str(corpus[5]["text_id"])  # The correct document ID for the excerpt we are testing against
print("Correct document ID:", correct_doc_id)


Detecting verbatim excerpt in corpus based on number of shared fingerprints
Scores: {'59569': 0.9999999995360956}
Correct document ID: 59569


Unsurprisingly, the containment score for the correct document ID is very high, indicating that the excerpt is indeed very likely to be present in that document.

### Near-verbatim detection 

We can check what happens if there are small discrepancies between the excerpt and the text in the corpus. Those discrepancies are highly common, and may be due to text standardisation, presence of OCR or formatting errors, or boilerplate, etc. Here, we apply some changes to the spelling and remove page/poem numbers:

In [23]:
import re

near_verbatim = (excerpt
    .replace("cc", "c")    # accolti→acolti, accosti→acosti, Ricciardetto→Riciardetto
    .replace("ss", "s")    # s'assicura→s'asicura, professione→profesione
    .replace("gli", "li")  # archaic Italian: "gli sparvieri" → "li sparvieri"
    .replace("gn", "ni")   # phonetic: segnano→seniano (gn=/ɲ/ → ni)
)

near_verbatim = re.sub(r"^\s*\[?\d+\]?\s*$", "", near_verbatim, flags=re.MULTILINE)

To derive the best alignment between this modified excerpt and the original book, one can use the ready-made local aligner implemented in <font face="Century Gothic, sans-serif"><b>Find<span style="color:#4A7A4A;">My</span>Text</span></b></font>, based on an efficient implementation of the Smith-Waterman algorithm : 

In [ ]:
from findmytext import oracle
alignment = oracle.align(near_verbatim, str(corpus[5]["text"]))
alignment.show()


The detector remains robust to those changes, and still detects the excerpt as being contained in the original text, although with a lower score, as can be seen below:

In [28]:
print("Detecting near-verbatim excerpt in corpus")
near_verbatim_scores = baseline_detector.get_containment_scores(near_verbatim)
print("Containment scores:", near_verbatim_scores)

Detecting near-verbatim excerpt in corpus
Containment scores: {'59569': 0.9429331688008555}


However, the `NbSharedFingerprintsDetector` has a major flaw: its score only depends on the presence of shared fingerprints, but it ignores their _order_. This means that a text in which lines are jumbled will still receive a high score, although the excerpt no longer represents a contiguous fragment of the original text: 

In [29]:
print("Detecting excerpt with all lines shuffled")
import random
lines = excerpt.split("\n")
random.shuffle(lines)
jumbled_excerpt = "\n".join(lines)

jumbled_scores = baseline_detector.get_containment_scores(jumbled_excerpt)
print("Containment scores:", jumbled_scores)

Detecting excerpt with all lines shuffled
Containment scores: {'59569': 0.406441970760905}


This stands in sharp contrast with the outputs of the local aligner, which tells us that the excerpt is no longer well aligned to the original text:

In [30]:
alignment = oracle.align(jumbled_excerpt, str(corpus[5]["text"]))
alignment.show()

### Detecting fingerprint chains

Fortunately, this shortcoming can be addressed by explicitly searching for _chains_ of fingerprints that are shared between the excerpt and a document in the corpus. As explained in our paper, detecting the presence of such chains can be done through a clustering approach, which we implement in `FingerprintChainDetector`:

In [31]:
chain_detector = detectors.FingerprintChainDetector("gutenberg_index")

print("Detecting verbatim excerpt in corpus based on the detection of fingerprint chains:")
score_correct = chain_detector.get_containment_scores(excerpt)
print("Containment scores:", score_correct)

print("Detecting near-verbatim excerpt in corpus based on the detection of fingerprint chains:")
score_near = chain_detector.get_containment_scores(near_verbatim)
print("Containment scores:", score_near)

Detecting verbatim excerpt in corpus based on the detection of fingerprint chains:
Containment scores: {'59569': 0.9999999995360956}
Detecting near-verbatim excerpt in corpus based on the detection of fingerprint chains:
Containment scores: {'59569': 0.9429331688008555}


As we can see, the `FingerprintChainDetector` provides similar high containment scores for both the verbatim and near-verbatim excerpts. However, it is no longer fooled by texts in which fingerprints appear in a different order than the original text:

In [32]:
print("Detecting shuffled excerpt in corpus based on the detection of fingerprint chains:")
score_shuffled = chain_detector.get_containment_scores(jumbled_excerpt)
print("Containment scores:", score_shuffled)

Detecting shuffled excerpt in corpus based on the detection of fingerprint chains:
Containment scores: {'59569': 5.4442763434649494e-05}


Contact us if you have any questions or would like to contribute to the project in one way or another. And do not hesitate to reach out and share how you are using **FindMyText** in your research or projects. We would love to hear from you :-)